IMPORTS AND SETUP

In [137]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

original_df = train_df.copy()
test_df_original = test_df.copy()

print("Data loaded successfully!")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Data loaded successfully!
Train shape: (9864, 19)
Test shape: (2466, 18)


EXPLORE DATA

In [138]:
print("=" * 50)
print("DATA EXPLORATION")
print("=" * 50)

print("\nColumns in dataset:")
print(train_df.columns.tolist())

print("\nData types:")
print(train_df.dtypes)

print("\nMissing values:")
missing = train_df.isnull().sum()
print(missing[missing > 0])

print("\nTarget distribution:")
print(train_df['Revenue'].value_counts())
print(train_df['Revenue'].value_counts(normalize=True) * 100)

X = train_df.drop('Revenue', axis=1)
y = train_df['Revenue'].astype(int)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical columns ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

DATA EXPLORATION

Columns in dataset:
['Session_ID', 'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

Data types:
Session_ID                   int64
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                       object
OperatingSystems             int64
Browser                      int64
Region                     float64
TrafficType                float64
VisitorType                 object
Weekend                       bool
Revenue      

PREPROCESSING PIPELINE

In [139]:
print("=" * 50)
print("PREPROCESSING PIPELINE")
print("=" * 50)

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

print("Preprocessor created successfully!")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

print("\nApplying preprocessing...")
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

print(f"Training data processed: {X_train_processed.shape}")
print(f"Validation data processed: {X_val_processed.shape}")

row_retention = (X_train_processed.shape[0] / X.shape[0]) * 100
print(f"\nRows retained: {row_retention:.2f}%")
if row_retention >= 90:
    print("PASSED: >= 90%")
else:
    print("FAILED: Need >= 90%")

processed_df = pd.DataFrame(X_train_processed)
processed_df['Revenue'] = y_train.values

print("processed_df created successfully!")

PREPROCESSING PIPELINE
Preprocessor created successfully!
Training set: 7891 samples
Validation set: 1973 samples

Applying preprocessing...
Training data processed: (7891, 26)
Validation data processed: (1973, 26)

Rows retained: 80.00%
FAILED: Need >= 90%
processed_df created successfully!


TRAIN LOGISTIC REGRESSION

In [140]:
print("=" * 50)
print("MODEL TRAINING")
print("=" * 50)

model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced',
    C=1.0,
    solver='lbfgs'
)

print("Training model...")
model.fit(X_train_processed, y_train)
print("Model training complete!")

y_val_pred = model.predict(X_val_processed)
accuracy = accuracy_score(y_val, y_val_pred)

print(f"\nValidation Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

cv_scores = cross_val_score(model, X_train_processed, y_train, cv=5)
print(f"\nCross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

feature_names = num_cols.copy()

for col in cat_cols:
    try:
        encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
        categories = encoder.categories_[list(cat_cols).index(col)]
        feature_names.extend([f"{col}_{cat}" for cat in categories[1:]])
    except:
        pass

coef_df = pd.DataFrame({
    'feature': feature_names[:len(model.coef_[0])],
    'coefficient': model.coef_[0],
    'abs_coef': abs(model.coef_[0])
}).sort_values('abs_coef', ascending=False)

print("\nTop 10 Most Important Features:")
print(coef_df.head(10))

MODEL TRAINING
Training model...
Model training complete!

Validation Accuracy: 0.8561

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.89      0.91      1617
           1       0.58      0.71      0.64       356

    accuracy                           0.86      1973
   macro avg       0.76      0.80      0.78      1973
weighted avg       0.87      0.86      0.86      1973


Cross-validation scores: [0.83280557 0.83079848 0.84030418 0.84347275 0.84283904]
Mean CV accuracy: 0.8380 (+/- 0.0105)

Top 10 Most Important Features:
                          feature  coefficient  abs_coef
9                      PageValues     1.765726  1.765726
16                      Month_Feb    -1.000762  1.000762
21                      Month_Nov     0.500334  0.500334
15                      Month_Dec    -0.402965  0.402965
20                      Month_May    -0.343923  0.343923
18                     Month_June    -0.331775  0.331775
8        

GENERATE TEST PREDICTIONS

In [141]:
print("=" * 50)
print("GENERATING PREDICTIONS")
print("=" * 50)

print("Processing test data...")

X_test = test_df.drop('Session_ID', axis=1).copy()

train_columns = X.columns.tolist()

for col in train_columns:
    if col not in X_test.columns:
        print(f"Adding missing column: {col}")
        X_test[col] = 0

X_test = X_test[train_columns]

print(f"Test data shape after alignment: {X_test.shape}")

X_test_processed = preprocessor.transform(X_test)
print(f"Test data processed: {X_test_processed.shape}")

print("Making predictions...")
predictions = model.predict(X_test_processed)
predictions = predictions.astype(bool)

print(f"Predictions generated: {len(predictions)}")
print(f"\nPrediction distribution:")
print(pd.Series(predictions).value_counts())
print(f"True (1): {sum(predictions)}")
print(f"False (0): {len(predictions) - sum(predictions)}")

GENERATING PREDICTIONS
Processing test data...
Adding missing column: Session_ID
Test data shape after alignment: (2466, 18)
Test data processed: (2466, 26)
Making predictions...
Predictions generated: 2466

Prediction distribution:
False    2386
True       80
Name: count, dtype: int64
True (1): 80
False (0): 2386


CREATE SUBMISSION

In [142]:
print("=" * 50)
print("CREATING SUBMISSION")
print("=" * 50)

original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (processed_rows / original_rows) * 100

final_model = model

if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_

if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],
    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat([checkpoints, prediction_output], ignore_index=True)
submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")
print("\nCheckpoint Information:")
print(checkpoints)
print(f"\nSubmission shape: {submission.shape}")
print(f"First 5 predictions:")
print(prediction_output.head(5))

print("\nVERIFICATION:")
print(f"Total rows in submission: {len(submission)}")
print(f"Checkpoint rows: {len(checkpoints)}")
print(f"Prediction rows: {len(prediction_output)}")
print(f"Row retention: {row_retained_percent:.2f}% >= 90%: {'PASS' if row_retained_percent >= 90 else 'FAIL'}")
print(f"Model name: {model_name} (should be LogisticRegression)")
print(f"Missing values in processed: {processed_missing} (should be 0)")
print(f"Predictions are boolean: {predictions.dtype}")

CREATING SUBMISSION
submission.csv created successfully!

Checkpoint Information:
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                7891
4      original_columns                  19
5     processed_columns                  27
6  row_retained_percent                80.0
7            model_name  LogisticRegression

Submission shape: (2474, 2)
First 5 predictions:
       id  value
0  106094  False
1  111845  False
2  106794  False
3  103444  False
4  106833  False

VERIFICATION:
Total rows in submission: 2474
Checkpoint rows: 8
Prediction rows: 2466
Row retention: 80.00% >= 90%: FAIL
Model name: LogisticRegression (should be LogisticRegression)
Missing values in processed: 0 (should be 0)
Predictions are boolean: bool


RETRAIN ON FULL DATA

In [143]:
print("=" * 50)
print("RETRAINING ON FULL DATASET")
print("=" * 50)

X_full_processed = preprocessor.fit_transform(X)
y_full = y

print(f"Full data processed: {X_full_processed.shape}")

model_full = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced',
    C=1.0,
    solver='lbfgs'
)

print("Training on full dataset...")
model_full.fit(X_full_processed, y_full)
print("Model training complete!")

model = model_full

processed_df = pd.DataFrame(X_full_processed)
processed_df['Revenue'] = y_full.values

row_retention = (X_full_processed.shape[0] / X.shape[0]) * 100
print(f"\nRows retained: {row_retention:.2f}%")
if row_retention >= 90:
    print("PASSED: >= 90%")
else:
    print("FAILED: Need >= 90%")

print(f"processed_df shape: {processed_df.shape}")

RETRAINING ON FULL DATASET
Full data processed: (9864, 26)
Training on full dataset...
Model training complete!

Rows retained: 100.00%
PASSED: >= 90%
processed_df shape: (9864, 27)


REGENERATE PREDICTIONS

In [144]:
print("=" * 50)
print("REGENERATING PREDICTIONS WITH FULL MODEL")
print("=" * 50)

X_test = test_df.drop('Session_ID', axis=1).copy()

train_columns = X.columns.tolist()

for col in train_columns:
    if col not in X_test.columns:
        print(f"Adding missing column: {col}")
        X_test[col] = 0

X_test = X_test[train_columns]
X_test_processed = preprocessor.transform(X_test)

predictions = model.predict(X_test_processed)
predictions = predictions.astype(bool)

print(f"Predictions generated: {len(predictions)}")
print(f"\nPrediction distribution:")
print(pd.Series(predictions).value_counts())
print(f"True (1): {sum(predictions)}")
print(f"False (0): {len(predictions) - sum(predictions)}")

REGENERATING PREDICTIONS WITH FULL MODEL
Adding missing column: Session_ID
Predictions generated: 2466

Prediction distribution:
False    2290
True      176
Name: count, dtype: int64
True (1): 176
False (0): 2290


In [145]:
print("=" * 50)
print("FIXING CHECKPOINT VALUES")
print("=" * 50)

processed_rows = processed_df.shape[0]
original_rows = original_df.shape[0]

row_retained_percent = (processed_rows / original_rows) * 100

print(f"Original rows: {original_rows}")
print(f"Processed rows: {processed_rows}")
print(f"Row retention: {row_retained_percent:.2f}%")

if row_retained_percent >= 90:
    print("PASSED: >= 90%")
else:
    print("FAILED: Need >= 90%")

checkpoints = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],
    "value": [
        original_df.isnull().sum().sum(),
        processed_df.isnull().sum().sum(),
        original_rows,
        processed_rows,
        original_df.shape[1],
        processed_df.shape[1],
        round(row_retained_percent, 2),
        model.__class__.__name__
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat([checkpoints, prediction_output], ignore_index=True)
submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv recreated successfully!")
print("\nCheckpoint Information:")
print(checkpoints)
print(f"\nRow retention: {row_retained_percent:.2f}%")
print("PASSED" if row_retained_percent >= 90 else "FAILED")

FIXING CHECKPOINT VALUES
Original rows: 9864
Processed rows: 9864
Row retention: 100.00%
PASSED: >= 90%

submission.csv recreated successfully!

Checkpoint Information:
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  27
6  row_retained_percent               100.0
7            model_name  LogisticRegression

Row retention: 100.00%
PASSED


check  

In [146]:
print("=" * 50)
print("YOUR ACCURACY SCORES")
print("=" * 50)

print("VALIDATION ACCURACY:")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nCROSS-VALIDATION ACCURACY:")
print(f"Mean CV: {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)")
print(f"CV Scores: {cv_scores}")

print("\nDATA SUMMARY:")
print(f"Training rows: {len(y)}")
print(f"Validation rows: {len(y_val)}")
print(f"Test predictions: {len(predictions)}")
print(f"Row retention: {row_retained_percent:.2f}%")

print("\nPREDICTION DISTRIBUTION ON TEST:")
true_count = sum(predictions)
false_count = len(predictions) - true_count
print(f"True (Purchase): {true_count} ({true_count/len(predictions)*100:.1f}%)")
print(f"False (No Purchase): {false_count} ({false_count/len(predictions)*100:.1f}%)")


YOUR ACCURACY SCORES
VALIDATION ACCURACY:
Accuracy: 0.8561 (85.61%)

CROSS-VALIDATION ACCURACY:
Mean CV: 0.8380 (83.80%)
CV Scores: [0.83280557 0.83079848 0.84030418 0.84347275 0.84283904]

DATA SUMMARY:
Training rows: 9864
Validation rows: 1973
Test predictions: 2466
Row retention: 100.00%

PREDICTION DISTRIBUTION ON TEST:
True (Purchase): 176 (7.1%)
False (No Purchase): 2290 (92.9%)
